# 🦙 Vuln Auditor — **CodeLlama-7B QLoRA** (sinh lời giải: Verdict + Nguyên nhân + Fix)
Chuyển từ CodeBERT (chỉ phân loại) sang **CodeLlama-7B** fine-tune bằng **QLoRA 4-bit**, huấn luyện theo kiểu **sinh giải thích** để mô hình *hiểu cội nguồn* lỗ hổng chứ không học lối tắt.

**Kỹ thuật giúp "hiểu cội nguồn":**
| Kỹ thuật | Tác dụng |
|---|---|
| **Rationale-SFT** | Không chỉ đoán nhãn — model phải xuất **Verdict + loại lỗ hổng + NGUYÊN NHÂN GỐC + cách sửa** (dùng `description/remediation/analysis` trong rag_knowledge; smartbug chỉ có category → map sang câu giải thích chuẩn) |
| **CodeLlama context 16k** | Đưa **cả contract** vào 1 lượt (max_seq 2048) → bỏ được chunk/MIL |
| **QLoRA 4-bit** | 7B fine-tune vừa 1×T4 16GB |
| **Loss chỉ trên phần trả lời** | Prompt bị mask (-100), chỉ học sinh lời giải |
| **Split chống rò rỉ + loại near-dup + baseline TF-IDF** | Số liệu **trung thực** (rút kinh nghiệm CodeBERT bị thổi điểm do near-dup) |
| **Đánh giá verdict-scoring** | So log-prob "Safe" vs "Vulnerable" → có xác suất + AUC + ngưỡng; kèm demo sinh lời giải |

> **Kaggle:** ① *Add Input → Models* → thêm **CodeLlama 7B** (không cần token). ② *Add Input → Datasets* → **rag_knowledge.jsonl**. ③ Bật GPU. ④ Chạy cell cài đặt → **Restart Kernel** → Run All.

## ⚙️ 0. Cài đặt — chạy rồi **RESTART KERNEL**

In [ ]:
# Bộ 4.x ổn định + bitsandbytes cho QLoRA 4-bit.
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "peft==0.13.2" "accelerate==1.0.1" "datasets==3.0.1" "bitsandbytes>=0.43.0" scikit-learn sentencepiece
import importlib
for pkg in ["transformers", "peft", "accelerate", "bitsandbytes", "torch"]:
    try: print(f"  {pkg}: {getattr(importlib.import_module(pkg), '__version__', '?')}")
    except Exception as e: print(f"  {pkg}: !! {e}")
print("\n🔴 BẮT BUỘC: Run -> Restart Kernel, rồi Run All.")

## 📦 1. Cấu hình

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # 7B 4-bit vừa 1×T4; tránh Trainer bật DataParallel khi Kaggle cấp 2 GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json, re, glob, hashlib, random, collections
import numpy as np, torch

import transformers
assert int(transformers.__version__.split(".")[0]) < 5, "CHƯA restart kernel! Restart rồi Run All."
print("transformers:", transformers.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

HF_FALLBACK = "NousResearch/CodeLlama-7b-hf"   # mirror ungated (khỏi xin quyền/token); thay được: "deepseek-ai/deepseek-coder-6.7b-base"
MAX_SEQ_LEN = 2048          # CodeLlama 16k; 2048 phủ đa số contract (giảm còn 1024 nếu thiếu thời gian)
LABELS = ["Safe", "Vulnerable"]

# QLoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

# Train (T4: batch 1 + accum 16 = effective 16; giảm MAX_SEQ_LEN nếu OOM/chậm)
EPOCHS       = 2
TRAIN_BS     = 1
EVAL_BS      = 1
GRAD_ACCUM   = 16
LR           = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0
MAX_NEW_TOKENS = 200

# Đánh giá trung thực
DROP_LEAKED = True          # loại mẫu val/test gần-trùng train (cosine TF-IDF > ngưỡng)
LEAK_SIM    = 0.9

SAVE_DIR = "./codellama-vuln-lora"
print("MAX_SEQ_LEN:", MAX_SEQ_LEN, "| QLoRA r:", LORA_R, "| epochs:", EPOCHS)

## 📂 2. Nạp dữ liệu + từ điển nguyên nhân/cách sửa theo loại lỗ hổng

In [ ]:
def find_data_path():
    for pat in ["/kaggle/input/**/rag_knowledge.jsonl", "rag_knowledge.jsonl",
                "data/clean_dataset/rag_knowledge.jsonl"]:
        h = sorted(glob.glob(pat, recursive=True))
        if h: return h[0]
    raise FileNotFoundError("Hãy Add Input -> Datasets -> rag_knowledge.jsonl")

rows = [json.loads(l) for l in open(find_data_path(), encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABELS]
for r in rows:
    r.setdefault("categories", []); r.setdefault("source", "unknown")
print("Mẫu:", len(rows), "| theo nguồn:", dict(collections.Counter(r["source"] for r in rows)))

# Câu giải thích NGUYÊN NHÂN GỐC chuẩn cho từng loại (dùng khi mẫu không có description sẵn, vd smartbug)
CATEGORY_CAUSE = {
 "reentrancy": "An external call is made before internal state is updated, letting the callee re-enter and repeat the withdrawal (violates checks-effects-interactions).",
 "arithmetic": "Integer arithmetic can overflow or underflow without checks, corrupting balances or bypassing limits.",
 "access_control": "A state-changing function lacks proper owner/role authorization, so any caller can invoke it.",
 "denial_service": "Control flow can be forced into a stuck state (unbounded loop, failing external call, or gas exhaustion), blocking legitimate use.",
 "front_running": "The outcome depends on transaction ordering; an attacker watching the mempool can insert a transaction first to profit.",
 "time_manipulation": "Critical logic relies on block.timestamp/now, which miners can nudge within a window.",
 "unchecked_low_calls": "The return value of a low-level call/send/delegatecall is not checked, so silent failures are ignored.",
 "bad_randomness": "Randomness is derived from predictable on-chain values (blockhash/timestamp), so it can be manipulated.",
 "other": "The contract contains a security weakness flagged during audit.",
}
CATEGORY_FIX = {
 "reentrancy": "Apply checks-effects-interactions and/or a reentrancy guard; update state before external calls.",
 "arithmetic": "Use checked arithmetic (Solidity >=0.8) or SafeMath and validate bounds.",
 "access_control": "Add onlyOwner/role modifiers and verify msg.sender before sensitive actions.",
 "denial_service": "Avoid unbounded loops, use pull-over-push payments, and handle external-call failures.",
 "front_running": "Use commit-reveal, slippage bounds, or minimize ordering dependence.",
 "time_manipulation": "Avoid timestamp for critical randomness/decisions; tolerate miner drift.",
 "unchecked_low_calls": "Check the boolean return of low-level calls and revert on failure.",
 "bad_randomness": "Use a secure randomness source (e.g., Chainlink VRF) instead of on-chain values.",
 "other": "Apply the standard mitigation for the identified weakness.",
}

def build_target(item):
    if item["label"] == "Safe":
        return ("Verdict: Safe\n"
                "The contract handles state updates, external calls, access control and arithmetic "
                "without an exploitable flaw.")
    cats = item.get("categories") or []
    title = item.get("title") or (", ".join(cats) if cats else "unspecified vulnerability")
    cause = (item.get("description") or item.get("analysis") or "").strip()
    if not cause:
        cause = " ".join(CATEGORY_CAUSE.get(c, "") for c in cats).strip() or CATEGORY_CAUSE["other"]
    fix = (item.get("remediation") or "").strip()
    if not fix:
        fix = " ".join(CATEGORY_FIX.get(c, "") for c in cats).strip() or CATEGORY_FIX["other"]
    cause = re.sub(r"\s+", " ", cause)[:600]; fix = re.sub(r"\s+", " ", fix)[:400]
    return f"Verdict: Vulnerable\nVulnerability: {title}\nRoot cause: {cause}\nFix: {fix}"

print("\n--- Ví dụ target (Vulnerable) ---\n", build_target(next(r for r in rows if r["label"]=="Vulnerable")))

## ✂️ 3. Split chống rò rỉ + loại near-duplicate + baseline TF-IDF (trần lối tắt)

In [ ]:
def structural_sig(code):
    s = re.sub(r"\b[A-Za-z_]\w*\b", "X", code)
    return hashlib.md5(re.sub(r"\s+", "", s).encode("utf-8", "replace")).hexdigest()

n = len(rows)
groups = collections.defaultdict(list)
for i, r in enumerate(rows): groups[structural_sig(r["code"])].append(i)
gk = list(groups.keys()); random.Random(SEED).shuffle(gk)
nt = int(0.10*n); nvl = int(0.10*n)
te, va, tr = [], [], []
for k in gk:
    g = groups[k]
    if len(te)+len(g) <= nt: te += g
    elif len(va)+len(g) <= nvl: va += g
    else: tr += g
take = lambda ix: [rows[i] for i in ix]
train_raw, val_raw, test_raw = take(tr), take(va), take(te)

# --- baseline TF-IDF + loại near-dup rò rỉ (để số liệu trung thực) ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(token_pattern=r"[A-Za-z_]\w+", max_features=40000, ngram_range=(1,2), sublinear_tf=True)
Vtr = vec.fit_transform([r["code"] for r in train_raw])
def filter_leak(split_raw, name):
    if not DROP_LEAKED: return split_raw
    Vs = vec.transform([r["code"] for r in split_raw])
    keep = []
    for i, r in enumerate(split_raw):
        same = [j for j, t in enumerate(train_raw) if t["source"] == r["source"]]
        mx = cosine_similarity(Vs[i], Vtr[same]).max() if same else 0.0
        if mx < LEAK_SIM: keep.append(r)
    print(f"  {name}: giữ {len(keep)}/{len(split_raw)} (loại {len(split_raw)-len(keep)} near-dup >{LEAK_SIM})")
    return keep
print("Loại near-duplicate rò rỉ:")
val_raw = filter_leak(val_raw, "val"); test_raw = filter_leak(test_raw, "test")

ytr = [r["label"]=="Vulnerable" for r in train_raw]
clf = LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced").fit(Vtr, ytr)
Vte = vec.transform([r["code"] for r in test_raw]); yte = np.array([r["label"]=="Vulnerable" for r in test_raw])
pb = clf.predict_proba(Vte)[:,1]
print(f"\n[BASELINE TF-IDF] test acc={accuracy_score(yte, pb>=.5):.3f} F1m={f1_score(yte, pb>=.5, average='macro'):.3f} AUC={roc_auc_score(yte, pb):.3f}")
print("   -> CodeLlama cần VƯỢT rõ mốc này thì mới coi là 'hiểu' hơn túi-từ.")
for nm, d in [("train", train_raw), ("val", val_raw), ("test", test_raw)]:
    v = sum(x["label"]=="Vulnerable" for x in d)
    print(f"  {nm:5s}: {len(d):5d} | Vuln {v} Safe {len(d)-v}")

## 🔤 4. Nạp tokenizer + tạo prompt/target (mask loss ở prompt)

In [ ]:
def find_model():
    cands = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        try: mt = json.load(open(cfg)).get("model_type", "")
        except Exception: continue
        has_w = (glob.glob(d+"/*.safetensors") or glob.glob(d+"/pytorch_model*.bin")
                 or glob.glob(d+"/*.index.json"))
        if mt == "llama" and has_w: cands.append(d)
    if cands:
        cands.sort(key=len); print("✅ Model (Kaggle Input):", cands[0]); return cands[0]
    print("⚠️ Không thấy model trong /kaggle/input -> fallback HF:", HF_FALLBACK,
          "(cần chấp nhận license + HF_TOKEN nếu bị gate)")
    return HF_FALLBACK

MODEL_PATH = find_model()
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "right"

PRE  = ("[INST] You are a smart-contract security auditor. Analyze the Solidity code and decide if it is "
        "Safe or Vulnerable. If Vulnerable, name the vulnerability type, explain the ROOT CAUSE, and give a fix.\n\n"
        "```solidity\n")
POST = "\n```\n[/INST] "
PRE_IDS  = tok(PRE, add_special_tokens=True).input_ids     # +BOS
POST_IDS = tok(POST, add_special_tokens=False).input_ids

def encode(item):
    tgt_ids  = tok(build_target(item), add_special_tokens=False).input_ids + [tok.eos_token_id]
    budget   = MAX_SEQ_LEN - len(PRE_IDS) - len(POST_IDS) - len(tgt_ids)
    if budget < 16:
        tgt_ids = tgt_ids[:max(1, MAX_SEQ_LEN - len(PRE_IDS) - len(POST_IDS) - 16)] + [tok.eos_token_id]
        budget = 16
    code_ids = tok(item["code"], add_special_tokens=False).input_ids[:budget]
    input_ids = PRE_IDS + code_ids + POST_IDS + tgt_ids
    labels    = [-100]*(len(PRE_IDS)+len(code_ids)+len(POST_IDS)) + tgt_ids
    return {"input_ids": input_ids, "labels": labels}

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, items): self.d = [encode(it) for it in items]
    def __len__(self): return len(self.d)
    def __getitem__(self, i): return self.d[i]

print("Đang tokenize..."); train_ds, val_ds = SFTDataset(train_raw), SFTDataset(val_raw)
lens = [len(x["input_ids"]) for x in train_ds.d]
print(f"Train {len(train_ds)} | seq len: median {int(np.median(lens))} p90 {int(np.percentile(lens,90))} max {max(lens)}")

PAD = tok.pad_token_id
def collate(fs):
    m = max(len(f["input_ids"]) for f in fs)
    ii, ll, am = [], [], []
    for f in fs:
        p = m - len(f["input_ids"])
        ii.append(f["input_ids"] + [PAD]*p); ll.append(f["labels"] + [-100]*p); am.append([1]*len(f["input_ids"]) + [0]*p)
    return {"input_ids": torch.tensor(ii), "attention_mask": torch.tensor(am), "labels": torch.tensor(ll)}

## 🦙 5. Nạp CodeLlama-7B 4-bit + gắn QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb,
                                             device_map={"": 0}, torch_dtype=torch.float16)  # TRỌN trên 1 GPU (tránh lỗi 2 GPU)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

## 🏋️ 6. Huấn luyện (QLoRA SFT)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
args = TrainingArguments(
    output_dir="./cl-out", num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine", fp16=True,
    optim="paged_adamw_8bit", logging_steps=25,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False, save_total_limit=1,
    report_to="none", seed=SEED, remove_unused_columns=False,
    dataloader_pin_memory=False)   # grad-checkpointing đã bật trong prepare_model_for_kbit_training
# Kaggle cấp 2×T4 nhưng model nằm trọn 1 GPU -> chặn Trainer tự bọc nn.DataParallel (gây lỗi scatter).
model.is_parallelizable = True
model.model_parallel = True
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.args._n_gpu = 1   # buộc dùng 1 GPU (nn.DataParallel bị bỏ qua khi n_gpu<=1)
print("🚀 Train (7B QLoRA — có thể ~1.5-3h trên T4)..."); trainer.train(); print("✅ Xong.")

## 📊 7. Đánh giá — verdict scoring (Safe vs Vulnerable) + per-source

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score, accuracy_score
model.eval()
SAFE_ID = tok(" Safe", add_special_tokens=False).input_ids[0]
VULN_ID = tok(" Vulnerable", add_special_tokens=False).input_ids[0]
VERD_IDS = tok("Verdict:", add_special_tokens=False).input_ids   # khớp prefix target lúc train

@torch.no_grad()
def vuln_prob(code):
    budget = MAX_SEQ_LEN - len(PRE_IDS) - len(POST_IDS) - len(VERD_IDS) - 4
    code_ids = tok(code, add_special_tokens=False).input_ids[:max(16, budget)]
    ids = PRE_IDS + code_ids + POST_IDS + VERD_IDS
    x = torch.tensor([ids]).to(model.device)
    lg = model(x).logits[0, -1].float()
    two = torch.softmax(torch.stack([lg[SAFE_ID], lg[VULN_ID]]), 0)
    return two[1].item()

def probs_for(split):
    out = []
    for i, r in enumerate(split):
        out.append(vuln_prob(r["code"]))
        if (i+1) % 100 == 0: print(f"  ...{i+1}/{len(split)}")
    return np.array(out)

print("Chấm VAL..."); vp = probs_for(val_raw); vy = np.array([r["label"]=="Vulnerable" for r in val_raw])
best_t, best_f1 = 0.5, -1
for t in np.linspace(0.05, 0.95, 91):
    f = f1_score(vy, vp>=t, average="macro")
    if f > best_f1: best_f1, best_t = f, float(t)
print(f"Ngưỡng tối ưu (VAL) = {best_t:.3f} | F1-macro(val) = {best_f1:.4f}")

print("Chấm TEST..."); tp = probs_for(test_raw); ty = np.array([r["label"]=="Vulnerable" for r in test_raw])
for nm, t in [("0.50", 0.5), (f"{best_t:.3f}", best_t)]:
    pr = (tp>=t).astype(int)
    print("\n"+"="*58+f"\n📊 TEST @ {nm}\n"+"="*58)
    print(classification_report(ty, pr, target_names=LABELS, digits=4, zero_division=0))
    print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(ty, pr))
print(f"\nROC-AUC={roc_auc_score(ty, tp):.4f} PR-AUC={average_precision_score(ty, tp):.4f}")
src = [r["source"] for r in test_raw]; pr = (tp>=best_t).astype(int)
print("\n--- Per-source (test) ---")
for s in sorted(set(src)):
    ix=[i for i,x in enumerate(src) if x==s]
    auc = roc_auc_score(ty[ix], tp[ix]) if len(set(ty[ix]))>1 else float("nan")
    print(f"  {s:14s} n={len(ix):4d} acc={accuracy_score(ty[ix], pr[ix]):.3f} F1m={f1_score(ty[ix], pr[ix], average='macro'):.3f} AUC={auc:.3f}")

## 🔍 8. Sinh lời giải (kiểm tra mô hình có *hiểu cội nguồn* không)

In [ ]:
@torch.no_grad()
def analyze(code, max_new_tokens=MAX_NEW_TOKENS):
    code_ids = tok(code, add_special_tokens=False).input_ids[:MAX_SEQ_LEN-len(PRE_IDS)-len(POST_IDS)-max_new_tokens-8]
    ids = PRE_IDS + code_ids + POST_IDS
    x = torch.tensor([ids]).to(model.device)
    out = model.generate(x, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][len(ids):], skip_special_tokens=True).strip()

REENTRANCY = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool success,) = msg.sender.call{value: amount}("");
    require(success);
    balances[msg.sender] -= amount;
}
"""
print("### Ca reentrancy kinh điển:\n", analyze(REENTRANCY))
for r in test_raw[:3]:
    print("\n" + "="*60)
    print("GROUND TRUTH:", r["label"], "| source:", r["source"])
    print("MODEL:\n", analyze(r["code"]))

## 💾 9. Lưu adapter

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR); tok.save_pretrained(SAVE_DIR)
json.dump({"base_model": MODEL_PATH, "max_seq_len": MAX_SEQ_LEN, "threshold": best_t,
           "prompt_pre": PRE, "prompt_post": POST}, open(SAVE_DIR+"/infer_config.json","w"), indent=2)
print("Đã lưu adapter:", os.listdir(SAVE_DIR))